<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day5_7_(260602)_%EC%B1%8C%EB%A6%B0%EC%A7%80_%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile /content/spring-lab/simple-signup/build.gradle

plugins {
    id 'java'
    id 'org.springframework.boot' version '3.3.5'
    id 'io.spring.dependency-management' version '1.1.6'
}

group = 'com.example'
version = '0.0.1-SNAPSHOT'

java {
    toolchain {
        languageVersion = JavaLanguageVersion.of(17)
    }
}

repositories {
    mavenCentral()
}

dependencies {
    implementation 'org.springframework.boot:spring-boot-starter-web'
    implementation 'org.springframework.boot:spring-boot-starter-thymeleaf'
    implementation 'org.springframework.boot:spring-boot-starter-jdbc'

    runtimeOnly 'org.mariadb.jdbc:mariadb-java-client'

    testImplementation 'org.springframework.boot:spring-boot-starter-test'
}

tasks.named('test') {
    useJUnitPlatform()
}

Overwriting /content/spring-lab/simple-signup/build.gradle


In [2]:
%%writefile /content/spring-lab/simple-signup/src/main/resources/application.properties

server.port=3100

spring.datasource.url=jdbc:mariadb://localhost:3306/signup_lab
spring.datasource.username=testuser
spring.datasource.password=1234
spring.datasource.driver-class-name=org.mariadb.jdbc.Driver

spring.thymeleaf.cache=false

Overwriting /content/spring-lab/simple-signup/src/main/resources/application.properties


In [3]:
%%writefile /content/spring-lab/simple-signup/src/main/java/com/example/demo/dto/TaskForm.java

package com.example.demo.dto;

public class TaskForm {

    private String title;
    private String category;
    private String status;
    private String manager;
    private String description;

    public TaskForm() {
    }

    public String getTitle() {
        return title;
    }

    public String getCategory() {
        return category;
    }

    public String getStatus() {
        return status;
    }

    public String getManager() {
        return manager;
    }

    public String getDescription() {
        return description;
    }

    public void setTitle(String title) {
        this.title = title;
    }

    public void setCategory(String category) {
        this.category = category;
    }

    public void setStatus(String status) {
        this.status = status;
    }

    public void setManager(String manager) {
        this.manager = manager;
    }

    public void setDescription(String description) {
        this.description = description;
    }
}

Writing /content/spring-lab/simple-signup/src/main/java/com/example/demo/dto/TaskForm.java


In [4]:
%%writefile /content/spring-lab/simple-signup/src/main/java/com/example/demo/domain/Task.java

package com.example.demo.domain;

import java.time.LocalDateTime;

public class Task {

    private Long id;
    private String title;
    private String category;
    private String status;
    private String manager;
    private String description;
    private LocalDateTime createdAt;

    public Task(
            Long id,
            String title,
            String category,
            String status,
            String manager,
            String description,
            LocalDateTime createdAt
    ) {
        this.id = id;
        this.title = title;
        this.category = category;
        this.status = status;
        this.manager = manager;
        this.description = description;
        this.createdAt = createdAt;
    }

    public Long getId() {
        return id;
    }

    public String getTitle() {
        return title;
    }

    public String getCategory() {
        return category;
    }

    public String getStatus() {
        return status;
    }

    public String getManager() {
        return manager;
    }

    public String getDescription() {
        return description;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }
}

Writing /content/spring-lab/simple-signup/src/main/java/com/example/demo/domain/Task.java


In [5]:
%%writefile /content/spring-lab/simple-signup/src/main/java/com/example/demo/repository/TaskRepository.java

package com.example.demo.repository;

import com.example.demo.domain.Task;
import com.example.demo.dto.TaskForm;
import org.springframework.jdbc.core.JdbcTemplate;
import org.springframework.jdbc.core.RowMapper;
import org.springframework.stereotype.Repository;

import java.util.List;

@Repository
public class TaskRepository {

    private final JdbcTemplate jdbcTemplate;

    public TaskRepository(JdbcTemplate jdbcTemplate) {
        this.jdbcTemplate = jdbcTemplate;
    }

    private final RowMapper<Task> taskRowMapper = (rs, rowNum) -> new Task(
            rs.getLong("id"),
            rs.getString("title"),
            rs.getString("category"),
            rs.getString("status"),
            rs.getString("manager"),
            rs.getString("description"),
            rs.getTimestamp("created_at").toLocalDateTime()
    );

    public void save(TaskForm form) {
        String sql = """
                INSERT INTO tasks
                (title, category, status, manager, description)
                VALUES (?, ?, ?, ?, ?)
                """;

        jdbcTemplate.update(
                sql,
                form.getTitle(),
                form.getCategory(),
                form.getStatus(),
                form.getManager(),
                form.getDescription()
        );
    }

    public List<Task> findAll() {
        String sql = """
                SELECT
                    id,
                    title,
                    category,
                    status,
                    manager,
                    description,
                    created_at
                FROM tasks
                ORDER BY id DESC
                """;

        return jdbcTemplate.query(sql, taskRowMapper);
    }
}

Writing /content/spring-lab/simple-signup/src/main/java/com/example/demo/repository/TaskRepository.java


In [6]:
%%writefile /content/spring-lab/simple-signup/src/main/java/com/example/demo/service/TaskService.java

package com.example.demo.service;

import com.example.demo.domain.Task;
import com.example.demo.dto.TaskForm;
import com.example.demo.repository.TaskRepository;
import org.springframework.stereotype.Service;

import java.util.List;

@Service
public class TaskService {

    private final TaskRepository repository;

    public TaskService(TaskRepository repository) {
        this.repository = repository;
    }

    public void createTask(TaskForm form) {
        validate(form);

        repository.save(form);
    }

    public List<Task> findTasks() {
        return repository.findAll();
    }

    private void validate(TaskForm form) {

        if (form.getTitle() == null || form.getTitle().isBlank()) {
            throw new IllegalArgumentException("항목명을 입력해 주세요.");
        }

        if (form.getCategory() == null || form.getCategory().isBlank()) {
            throw new IllegalArgumentException("분류를 입력해 주세요.");
        }

        if (form.getStatus() == null || form.getStatus().isBlank()) {
            throw new IllegalArgumentException("상태를 입력해 주세요.");
        }

        if (form.getManager() == null || form.getManager().isBlank()) {
            throw new IllegalArgumentException("담당자를 입력해 주세요.");
        }
    }
}

Writing /content/spring-lab/simple-signup/src/main/java/com/example/demo/service/TaskService.java


In [10]:
%%writefile /content/spring-lab/simple-signup/src/main/java/com/example/demo/controller/TaskController.java

package com.example.demo.controller;

import com.example.demo.dto.TaskForm;
import com.example.demo.service.TaskService;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.*;

@Controller
public class TaskController {

    private final TaskService service;

    public TaskController(TaskService service) {
        this.service = service;
    }

    @GetMapping("/")
    public String home() {
        return "redirect:/tasks";
    }

    @GetMapping("/tasks")
    public String tasks(Model model) {

        model.addAttribute("taskForm", new TaskForm());
        model.addAttribute("tasks", service.findTasks());

        return "tasks";
    }

    @PostMapping("/tasks")
    public String createTask(@ModelAttribute TaskForm form, Model model) {

        try {
            service.createTask(form);

            return "redirect:/tasks";

        } catch (IllegalArgumentException e) {

            model.addAttribute("taskForm", form);
            model.addAttribute("tasks", service.findTasks());
            model.addAttribute("errorMessage", e.getMessage());

            return "tasks";
        }
    }
}

Overwriting /content/spring-lab/simple-signup/src/main/java/com/example/demo/controller/TaskController.java


In [8]:
%%writefile /content/spring-lab/simple-signup/src/main/resources/templates/tasks.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>작업 관리</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="page">

    <section class="card form-card">
        <h1>작업 등록</h1>

        <p class="description">
            작업 정보를 입력하면 DTO, Controller, Service, Repository를 거쳐
            MariaDB에 저장됩니다.
        </p>

        <div
                class="error-box"
                th:if="${errorMessage != null}"
                th:text="${errorMessage}">
            오류 메시지
        </div>

        <form action="/tasks" method="post" th:object="${taskForm}">

            <div class="field">
                <label>항목명</label>
                <input type="text"
                       th:field="*{title}"
                       placeholder="회원 관리 API 개발">
            </div>

            <div class="field">
                <label>분류</label>

                <select th:field="*{category}">
                    <option value="">선택</option>
                    <option value="기획">기획</option>
                    <option value="개발">개발</option>
                    <option value="테스트">테스트</option>
                    <option value="디자인">디자인</option>
                    <option value="운영">운영</option>
                </select>
            </div>

            <div class="field">
                <label>상태</label>

                <select th:field="*{status}">
                    <option value="">선택</option>
                    <option value="대기">대기</option>
                    <option value="진행중">진행중</option>
                    <option value="완료">완료</option>
                </select>
            </div>

            <div class="field">
                <label>담당자</label>

                <input type="text"
                       th:field="*{manager}"
                       placeholder="홍길동">
            </div>

            <div class="field">
                <label>설명</label>

                <textarea
                        th:field="*{description}"
                        placeholder="작업 설명 입력"></textarea>
            </div>

            <button type="submit">작업 등록</button>

        </form>
    </section>

    <section class="card list-card">

        <h2>작업 목록</h2>

        <p class="description">
            DB에서 조회된 데이터가 Task Domain 객체로 변환되어 출력됩니다.
        </p>

        <table>

            <thead>
            <tr>
                <th>ID</th>
                <th>항목명</th>
                <th>분류</th>
                <th>상태</th>
                <th>담당자</th>
                <th>설명</th>
                <th>등록일</th>
            </tr>
            </thead>

            <tbody>

            <tr th:each="task : ${tasks}">
                <td th:text="${task.id}">1</td>

                <td th:text="${task.title}">
                    작업명
                </td>

                <td th:text="${task.category}">
                    개발
                </td>

                <td th:text="${task.status}">
                    진행중
                </td>

                <td th:text="${task.manager}">
                    홍길동
                </td>

                <td th:text="${task.description}">
                    설명
                </td>

                <td th:text="${#temporals.format(task.createdAt, 'yyyy-MM-dd HH:mm:ss')}">
                    2026-01-01 10:00:00
                </td>
            </tr>

            </tbody>
        </table>
    </section>

</div>
</body>
</html>

Writing /content/spring-lab/simple-signup/src/main/resources/templates/tasks.html


In [11]:
%%writefile /content/spring-lab/simple-signup/src/main/resources/static/style.css

* {
    box-sizing: border-box;
}

body {
    margin: 0;
    font-family: Arial, "Noto Sans KR", sans-serif;
    background: #f5f7fb;
    color: #202124;
}

.page {
    width: 980px;
    margin: 40px auto;
    display: grid;
    gap: 24px;
}

.card {
    background: #ffffff;
    border: 1px solid #dadce0;
    border-radius: 18px;
    padding: 28px;
    box-shadow: 0 10px 28px rgba(60, 64, 67, 0.08);
}

h1,
h2 {
    margin-top: 0;
    letter-spacing: -0.04em;
}

.description {
    color: #5f6368;
    line-height: 1.6;
    margin-bottom: 24px;
}

form {
    display: grid;
    gap: 16px;
}

.field {
    display: grid;
    gap: 7px;
}

label {
    font-size: 14px;
    font-weight: 700;
    color: #5f6368;
}

input,
select,
textarea {
    width: 100%;
    border: 1px solid #dadce0;
    border-radius: 12px;
    padding: 12px 14px;
    outline: none;
    font-size: 14px;
    font-family: inherit;
    background: white;
}

input,
select {
    height: 44px;
}

textarea {
    min-height: 120px;
    resize: vertical;
}

input:focus,
select:focus,
textarea:focus {
    border-color: #1a73e8;
    box-shadow: 0 0 0 4px rgba(26, 115, 232, 0.12);
}

button {
    height: 44px;
    border: 0;
    border-radius: 12px;
    background: #1a73e8;
    color: white;
    font-weight: 700;
    cursor: pointer;
}

button:hover {
    background: #1558b0;
}

.error-box {
    padding: 12px 14px;
    margin-bottom: 18px;
    border-radius: 12px;
    background: #fce8e6;
    color: #d93025;
    border: 1px solid #fad2cf;
    font-weight: 700;
}

table {
    width: 100%;
    border-collapse: collapse;
}

th {
    text-align: left;
    color: #5f6368;
    font-size: 13px;
    padding: 12px 10px;
    border-bottom: 1px solid #dadce0;
}

td {
    padding: 13px 10px;
    border-bottom: 1px solid #eef0f3;
}

@media (max-width: 1000px) {
    .page {
        width: auto;
        margin: 20px;
    }

    table {
        display: block;
        overflow-x: auto;
        white-space: nowrap;
    }
}

Overwriting /content/spring-lab/simple-signup/src/main/resources/static/style.css
